In [ ]:
# Cell 0 — Install dependencies (Unsloth canonical Colab branch) + clone feat/corpo
#
# Mirrors the install logic from Unsloth's Qwen2.5_(3B)-GRPO notebook, cells 4+5:
#   https://raw.githubusercontent.com/unslothai/notebooks/main/nb/Qwen2.5_(3B)-GRPO.ipynb
#
# Why this exact shape (do not "simplify"):
#   - UNSLOTH_VLLM_STANDBY=1   → +30% context-length headroom (unsloth-specific)
#   - upgrade `uv` first       → uv's resolver is stricter than pip's; avoids the
#                                pip-picks-wrong-trl mistakes we hit on Path B
#   - GPU-aware vllm/triton    → T4 needs vllm==0.9.2 + triton==3.2.0; A100/L4/H100
#                                use vllm==0.15.1 + latest triton. Latest vllm
#                                doesn't work on T4 (silent crash on import).
#   - ONE uv-call bundle       → vllm + numpy + pil + torchvision + bitsandbytes +
#                                xformers + unsloth resolved together so versions
#                                stay consistent (no pip-installs-X-then-uv-overrides-Y).
#   - trl==0.22.2 --no-deps    → avoids TRL 0.24's vllm_ascend + mergekit imports.
#                                --no-deps so trl doesn't drag in transitive packages
#                                that would fight the unsloth-bundled versions.
#   - transformers==4.56.2     → matches what unsloth's wheel was built against.
#   - peft==0.17.1 --no-deps   → 0.18+ added _maybe_shard_state_dict_for_tp which
#                                imports transformers.integrations.tensor_parallel.
#                                EmbeddingParallel — that symbol doesn't exist
#                                until transformers 4.57+. Pinning to 0.17.1 (last
#                                pre-TP release, 2025-08-21) avoids the conflict.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab environment (local dev, RunPod, etc.): unsloth's simple path
    !pip install unsloth vllm
else:
    # Resolve currently-installed numpy + pillow versions to avoid churning them
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil   = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"

    # GPU-aware vllm + triton pinning
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except Exception:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq --no-deps peft==0.17.1

# Project-specific extras (not part of the canonical Unsloth recipe):
#   openai — DeepSeek V4-Pro judge HTTP client
!uv pip install -qqq "openai>=1.0.0"

# Clone the project at feat/corpo (private repo)
from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')
!rm -rf /content/sft
!git clone --branch feat/corpo --depth 1 \
    https://{GITHUB_PAT}@github.com/deepanathanrajendiran-hub/sft-code-review.git \
    /content/sft
!cp /content/sft/*.py /content/sft/pyproject.toml /content/
!cp -r /content/sft/tests /content/
os.chdir("/content")

!ls /content/corpo_*.py /content/swecare_*.py /content/ood_metrics.py
print("\nVersion check:")
import trl, vllm, torch, datasets, peft, transformers
print(f"  trl          : {trl.__version__}     (expected 0.22.2)")
print(f"  transformers : {transformers.__version__}    (expected 4.56.2)")
print(f"  vllm         : {vllm.__version__}     (T4=0.9.2, else=0.15.1)")
print(f"  torch        : {torch.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  peft         : {peft.__version__}    (expected 0.17.1)")

In [ ]:
# Cell 1 — Mount Drive, load secrets, verify v4 backup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

# v4 adapter paths — USER MUST ensure backup exists before this cell runs
V4_ADAPTER = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
V4_BACKUP  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup'
assert os.path.exists(V4_BACKUP + '/adapter_config.json'), \
    f"v4 backup missing! Create one BEFORE running: cp -r {V4_ADAPTER} {V4_BACKUP}"
print(f"v4 adapter:  {V4_ADAPTER}")
print(f"v4 backup:   {V4_BACKUP}")

In [ ]:
# Cell 2 — Load SWE-CARE dev (train) and test (eval), build base-sample cache (~5-10 min, $0.05)
# dev=7086 rows for CoRPO training; test=632 (after repo filter) reserved for OOD eval
import json, os, random

# Generate training prompts from dev split
!python /content/swecare_loader.py \
    --split dev \
    --output /content/ood_dev_prompts_raw.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl

# Sample 1500 dev rows (seeded) for actual training
with open('/content/ood_dev_prompts_raw.jsonl') as f:
    dev_rows = [json.loads(l) for l in f if l.strip()]
sample = random.Random(42).sample(dev_rows, min(1500, len(dev_rows)))
with open('/content/ood_train_prompts.jsonl', 'w') as f:
    for r in sample: f.write(json.dumps(r) + '\n')
print(f"dev pool: {len(dev_rows)}   training sample: {len(sample)}")

# Generate eval input from test split (separate, no overlap)
!python /content/swecare_loader.py \
    --split test \
    --output /content/ood_input.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl

# Diagnostic: how much repo-level overlap between training (dev) and eval (test)?
import json
def _read_repos(path):
    with open(path) as f:
        return {json.loads(l)['repo'] for l in f if l.strip()}

train_repos = _read_repos('/content/ood_train_prompts.jsonl')
eval_repos = _read_repos('/content/ood_input.jsonl')
overlap = train_repos & eval_repos
unique_to_eval = eval_repos - train_repos
print(f"[cell2] train repos:    {len(train_repos)}")
print(f"[cell2] eval repos:     {len(eval_repos)}")
print(f"[cell2] overlap:        {len(overlap)}/{len(eval_repos)} eval repos also in train ({100*len(overlap)/max(len(eval_repos),1):.0f}%)")
print(f"[cell2] eval-only:      {len(unique_to_eval)}")
print(f"[cell2] interpretation: {'high overlap — `novel PR in mostly-shared repos` measurement' if len(overlap)/max(len(eval_repos),1) > 0.5 else 'mostly-disjoint repos — closer to true OOD'}")

# Skip base cache build if cache exists AND covers every prompt in the current sample
import os, json
cache_path = '/content/cache/base_samples.jsonl'
build_cache = True
if os.path.exists(cache_path):
    with open(cache_path) as f:
        cached_ids = {json.loads(l)['instance_id'] for l in f if l.strip()}
    sample_ids = {r['instance_id'] for r in sample}
    missing = sample_ids - cached_ids
    if not missing:
        print(f"[cell2] base cache covers all {len(sample_ids)} sample prompts — skipping")
        build_cache = False
    else:
        print(f"[cell2] base cache missing {len(missing)} prompts (e.g. {next(iter(missing))!r}) — rebuilding")

if build_cache:
    os.system(
        'python /content/corpo_reward.py --build-base-cache '
        f'--input /content/ood_train_prompts.jsonl '
        f'--output {cache_path} '
        '--base-model unsloth/Qwen2.5-Coder-7B-Instruct'
    )

In [ ]:
# Cell 3 — Pre-training variance gate + auto-pick R_min for Run #3 (~10 min, $0.40)
#
# Run #3 calibrates R_min from the actual rollout reward distribution (CoRPO paper
# §3.1: "below median of correct rollouts, above max of incorrect"). The variance
# gate prints p25/p33/p40/p50 percentile candidates; this cell auto-extracts the
# p33 recommendation and stores it as R_MIN for Cell 4.
#
# If the histogram shows a clear bimodal trough between two humps, override R_MIN
# manually (set R_MIN = <your_value>) before running Cell 4.
import subprocess, re, sys

print("[cell3] running variance gate...")
result = subprocess.run(
    ["python", "/content/corpo_train.py", "--variance-gate-only",
     "--v4-adapter", V4_ADAPTER,
     "--v4-backup",  V4_BACKUP,
     "--train-prompts", "/content/ood_train_prompts.jsonl",
     "--base-cache", "/content/cache/base_samples.jsonl",
     "--output-dir", "/content/corpo-out"],
    capture_output=True, text=True,
)

# Show histogram + percentiles + verdict
print(result.stdout)
print(result.stderr, file=sys.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "[variance-gate] FAILED — reward signal too weak (mean within-group std < 0.10). "
        "Do not proceed to Cell 4. Investigate the reward function or base-sample cache."
    )

# Parse the p33 recommendation from stderr
m = re.search(r"recommended default: p33 = ([\d.]+)", result.stderr)
if not m:
    raise RuntimeError(
        "[variance-gate] PASSED but couldn't parse R_min recommendation. "
        "Check the printed output above and set R_MIN manually."
    )
R_MIN = float(m.group(1))

print()
print(f"[cell3] PASS — auto-selected R_MIN = {R_MIN}  (p33 of v4 rollout distribution)")
print(f"[cell3] If the histogram above looks bimodal, set R_MIN at the trough manually.")
print(f"[cell3] Otherwise, proceed to Cell 4.")

In [ ]:
# Cell 4 — Train CoRPO Run #3 (after Runs #1 and #2 both failed)
#
# Run #1 failures (R_min=0.5, kl_beta=0.04):
#   - R_min below group_mean=0.65 → CoRPO clip never engaged, degenerated to GRPO
#   - Length collapsed 1606→750 chars; -22pp pairwise vs v4
#
# Run #2 failures (R_min=0.70, kl_beta=0.02):
#   - R_min above group_mean=0.50 → every rollout clipped to negative advantage
#   - Length exploded 602→2642 chars; -52pp pairwise vs v4
#   - Length anchor was [1500, 4500] (training-time mean), not deployment (~600 chars)
#   - composite_reward scored raw rollout, not extracted review (train-eval skew)
#
# Run #3 Tier-1 fixes (literature-validated, 2026-05-26):
#   - R_min derived from variance-gate p33 (see Cell 3 — calibration, not guess)
#   - kl_beta=0.0          : CoRPO paper runs at beta=0; LoRA provides implicit reg
#   - loss_type="dr_grpo"  : Dr.GRPO normalization (arXiv:2503.20783) — fixes length bias
#   - composite_reward calls _extract_review first : matches deployment string
#   - length_sanity band [150, 1000] : anchored on v4 OOD median (~600 chars)
#   - checkpoint-every 75  : 5 checkpoints across ~375 steps (mid-eval at each)

!python /content/corpo_train.py \
    --v4-adapter {V4_ADAPTER} \
    --v4-backup {V4_BACKUP} \
    --train-prompts /content/ood_train_prompts.jsonl \
    --base-cache /content/cache/base_samples.jsonl \
    --output-dir /content/corpo-out \
    --checkpoint-sync-dir /content/drive/MyDrive/sft/corpo-out-run3 \
    --r-min-correct {R_MIN} \
    --kl-beta 0.0 \
    --learning-rate 5e-6 \
    --num-generations 8 \
    --prompts-per-step 4 \
    --max-new-tokens 2048 \
    --epochs 1 \
    --checkpoint-every 75 \
    --copy-to /content/drive/MyDrive/sft/code-reviewer-lora-v4-corpo-run3

In [ ]:
# Cell 5 — Mid-eval at each Run #3 checkpoint; auto-pick best for final OOD eval
#
# Iterates over every checkpoint synced to Drive (steps 75, 150, 225, 300, ~375),
# generates predictions on a fixed 50-sample subset of the OOD set, judges each
# pairwise vs v4 with V4-Pro, prints the win rate, and stores the best-performing
# checkpoint path as BEST_CHECKPOINT for Cell 6.
#
# Cost: ~$15 per checkpoint × ~5 checkpoints = ~$75 V4-Pro judge calls
# Time:  ~3 min vLLM gen + ~2 min judge per checkpoint = ~25 min total
#
# Early-kill criteria (apply by inspecting the printed win rates):
#   - step 75:  pairwise < 40%  → catastrophic; v4 already much better
#   - step 150: pairwise < 60%  → Run #3 is failing the same way as Runs #1/2
#   - step 225+: monotonically worsening → trajectory is bad; abort

import gc, json, random, sys, torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

sys.path.insert(0, '/content')
from run_ood_eval import _extract_review
from ood_metrics import deepseek_v4pro_pairwise_judge

# Config
N_SAMPLES = 50
CHECKPOINT_ROOT = '/content/drive/MyDrive/sft/corpo-out-run3'
V4_PREDS_PATH = '/content/drive/MyDrive/sft/ood_preds_v4.jsonl'

# Sample 50 v4 predictions (fixed seed → same prompts for every checkpoint, comparable)
random.seed(42)
with open(V4_PREDS_PATH) as f:
    all_v4 = [json.loads(l) for l in f if l.strip()]
eval_subset = random.sample(all_v4, min(N_SAMPLES, len(all_v4)))
print(f"[cell5] mid-eval subset: {len(eval_subset)} prompts from {V4_PREDS_PATH}")

# Find all checkpoints in Drive sync dir
checkpoint_dirs = sorted(
    [d for d in Path(CHECKPOINT_ROOT).iterdir()
     if d.is_dir() and d.name.startswith('checkpoint-')],
    key=lambda p: int(p.name.split('-')[1])
)
print(f"[cell5] found {len(checkpoint_dirs)} checkpoints: {[d.name for d in checkpoint_dirs]}")
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoints found in {CHECKPOINT_ROOT}. Did Cell 4 complete?")

# Build chat-templated prompts (same format as training + run_ood_eval)
SYSTEM_MSG = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."
USER_TEMPLATE = "Review the following code diff and provide feedback:\n```diff\n{diff}\n```"

tokenizer = AutoTokenizer.from_pretrained(V4_ADAPTER)
prompts = []
for row in eval_subset:
    msgs = [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": USER_TEMPLATE.format(diff=row['diff'][:12000])}]
    prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

# vLLM with LoRA swapping (avoids re-merging each checkpoint to disk)
llm = LLM(
    model='unsloth/Qwen2.5-Coder-7B-Instruct',
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    enable_lora=True,
    max_lora_rank=64,
)
sp = SamplingParams(temperature=0, max_tokens=4096, repetition_penalty=1.1)

def _judge_one(args):
    i, corpo_pred, v4_pred, diff = args
    return i, deepseek_v4pro_pairwise_judge(corpo_pred, v4_pred, diff, "")

results = {}
for ckpt_dir in checkpoint_dirs:
    step = int(ckpt_dir.name.split('-')[1])
    print(f"\n[mid-eval] === checkpoint-{step} ===")

    # Generate via LoRA swap (no re-merge)
    lora_req = LoRARequest(f"corpo-{step}", step, str(ckpt_dir))
    outputs = llm.generate(prompts, sp, lora_request=lora_req)
    corpo_preds = [_extract_review(o.outputs[0].text) for o in outputs]

    # Judge each pair (corpo=A, v4=B) in parallel
    work = [(i, corpo_preds[i], eval_subset[i].get('v4_pred', ''),
             eval_subset[i]['diff']) for i in range(len(corpo_preds))]
    verdicts = [None] * len(work)
    with ThreadPoolExecutor(max_workers=16) as executor:
        for i, v in executor.map(_judge_one, work):
            verdicts[i] = v

    wins = sum(1 for v in verdicts if v == 'A')
    losses = sum(1 for v in verdicts if v == 'B')
    ties = sum(1 for v in verdicts if v == 'TIE')
    win_rate = wins / len(verdicts)
    mean_chars = sum(len(p) for p in corpo_preds) / len(corpo_preds)

    results[step] = {
        'wins': wins, 'losses': losses, 'ties': ties,
        'win_rate': win_rate, 'mean_chars': mean_chars,
    }
    print(f"[mid-eval] step {step}: {wins}W/{losses}L/{ties}T  win_rate={win_rate*100:.1f}%  "
          f"mean_chars={mean_chars:.0f}")

# Cleanup vLLM so Cell 6's subprocess can claim the GPU
del llm
gc.collect()
torch.cuda.empty_cache()

# Pick best checkpoint (highest pairwise win rate)
best_step = max(results.keys(), key=lambda s: results[s]['win_rate'])
BEST_CHECKPOINT = f"{CHECKPOINT_ROOT}/checkpoint-{best_step}"

print(f"\n[mid-eval] summary:")
print(f"  {'step':>6} {'win%':>6} {'wins':>5} {'loss':>5} {'tie':>4} {'chars':>7}")
for step in sorted(results):
    r = results[step]
    marker = " <-best" if step == best_step else ""
    print(f"  {step:>6} {r['win_rate']*100:>5.1f}% {r['wins']:>5} {r['losses']:>5} "
          f"{r['ties']:>4} {r['mean_chars']:>7.0f}{marker}")

print(f"\n[mid-eval] BEST_CHECKPOINT = {BEST_CHECKPOINT}  (win rate {results[best_step]['win_rate']*100:.1f}%)")

# Sanity gate: if even the best checkpoint loses to v4 by >5pp, Run #3 has failed
if results[best_step]['win_rate'] < 0.45:
    print(f"\n[mid-eval] WARNING: best checkpoint wins only {results[best_step]['win_rate']*100:.1f}% "
          f"vs v4. Run #3 failed the same way as Runs #1/2. Skip Cell 6+7 and document negative result.")

# Persist raw results for journal entry
with open('/content/mid_eval_results_run3.json', 'w') as f:
    json.dump({str(k): v for k, v in results.items()}, f, indent=2)
print(f"[mid-eval] raw results saved to /content/mid_eval_results_run3.json")

In [ ]:
# Cell 6 — Verify chat_template parity, merge BEST_CHECKPOINT, generate predictions on full 632 OOD set
#
# Uses BEST_CHECKPOINT picked in Cell 5 (highest mid-eval pairwise win rate).
# Falls back to /content/corpo-out/final if Cell 5 was skipped.

# 6a. Verify v4 chat_template matches base — if not, run_ood_eval.py's assert will fire mid-run
from transformers import AutoTokenizer
v4_tok = AutoTokenizer.from_pretrained(V4_ADAPTER)
base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')
assert v4_tok.chat_template == base_tok.chat_template, \
    "v4 chat_template differs from base — patch run_ood_eval.py:135 to load tokenizer per-model"
del v4_tok, base_tok
print("[cell6] chat_template parity: OK")

# 6b. Determine which checkpoint to evaluate
try:
    _src = BEST_CHECKPOINT  # set by Cell 5
    print(f"[cell6] using BEST_CHECKPOINT from Cell 5: {_src}")
except NameError:
    _src = '/content/corpo-out/final'
    print(f"[cell6] Cell 5 skipped — falling back to {_src}")

# 6c. Merge corpo adapter (for vLLM eval — vLLM expects a full model)
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    'unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16
)
peft_model = PeftModel.from_pretrained(base, _src)
merged = peft_model.merge_and_unload()
merged.save_pretrained('/content/sft-v4corpo-merged-for-eval', safe_serialization=True)
AutoTokenizer.from_pretrained(V4_ADAPTER).save_pretrained('/content/sft-v4corpo-merged-for-eval')

del merged, peft_model, base
gc.collect(); torch.cuda.empty_cache()

# 6d. Generate v4-corpo predictions ONLY on the 632 OOD set (--skip-base saves ~30 min)
!python /content/run_ood_eval.py \
    --input /content/ood_input.jsonl \
    --output /content/ood_preds_v4corpo.jsonl \
    --v4-model /content/sft-v4corpo-merged-for-eval \
    --skip-base

In [ ]:
# Cell 7 — V4-Pro 3-vote re-baseline + post-corpo eval + Haiku cross-check + decision gate
import json

# 1. Recompute halluc_v4 from baseline preds under V4-Pro (skip pairwise — gate only needs halluc + per-domain)
!python /content/ood_metrics.py \
    --preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/ood_input.jsonl \
    --judge v4pro3vote \
    --skip-pairwise \
    --output /content/v4_baseline_v4pro.json

# 2. Merge v4_pred (from baseline preds) into v4corpo preds for the corpo-vs-v4 comparison
def _merge_preds(v4_path, corpo_path, out_path):
    v4_lookup = {}
    with open(v4_path) as f:
        for line in f:
            r = json.loads(line)
            v4_lookup[r['instance_id']] = r['v4_pred']
    with open(corpo_path) as f, open(out_path, 'w') as out:
        for line in f:
            c = json.loads(line)
            iid = c['instance_id']
            c['v4corpo_pred'] = c.pop('v4_pred', '')  # corpo run wrote it as v4_pred
            c['v4_pred'] = v4_lookup.get(iid, '')
            out.write(json.dumps(c) + '\n')

_merge_preds('/content/drive/MyDrive/sft/ood_preds_v4.jsonl',
             '/content/ood_preds_v4corpo.jsonl',
             '/content/ood_preds_merged.jsonl')

# 3. v4-corpo vs v4 under V4-Pro 3-vote (THE measurement)
!python /content/ood_metrics.py \
    --preds /content/ood_preds_merged.jsonl \
    --labels /content/ood_input.jsonl \
    --judge v4pro3vote \
    --pred-a-field v4corpo_pred \
    --pred-b-field v4_pred \
    --output /content/corpo_vs_v4_v4pro.json

# 4. Haiku cross-check on 100-prompt subset (Goodhart guard)
# Sample 100 random rows (seeded) for Haiku cross-check (NOT first 100 — that's grouped by dataset order)
import random
with open('/content/ood_preds_merged.jsonl') as f:
    all_rows = [l.strip() for l in f if l.strip()]
subset = random.Random(42).sample(all_rows, min(100, len(all_rows)))
with open('/content/ood_preds_merged_100.jsonl', 'w') as f:
    for r in subset: f.write(r + '\n')
!python /content/ood_metrics.py \
    --preds /content/ood_preds_merged_100.jsonl \
    --labels /content/ood_input.jsonl \
    --judge haiku \
    --pred-a-field v4corpo_pred \
    --pred-b-field v4_pred \
    --output /content/corpo_vs_v4_haiku.json

# 5. Decision gate verdict
!python /content/corpo_decision_gate.py \
    --v4-baseline-json /content/v4_baseline_v4pro.json \
    --corpo-eval-json  /content/corpo_vs_v4_v4pro.json \
    --haiku-cross-check-json /content/corpo_vs_v4_haiku.json \
    --variance-gate-passed